# Modul 5 · Interpretasi Hasil Analisis AI untuk Bisnis
**Pelatihan Machine Learning & Data Analysis**

Model yang akurat tapi tidak bisa dijelaskan = sulit dipakai mengambil keputusan. Modul ini melatih **3 keterampilan** menerjemahkan hasil AI:

1. **FAKTOR PENDORONG** — fitur apa yang paling memengaruhi prediksi? (*feature importance*) → jadi dasar kebijakan, bukan sekadar angka.
2. **NILAI BISNIS** — mengubah metrik teknis (akurasi, confusion matrix) menjadi **rupiah** dan risiko yang dipahami manajemen.
3. **KOMUNIKASI** — menyusun ringkasan eksekutif otomatis + memahami **batasan model** (apa yang *tidak* boleh disimpulkan).

> ⚠️ Prasyarat: notebook `00_generate_dataset.ipynb` sudah dijalankan.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

DIR = Path.cwd().parent

### Latih ulang model risiko kredit dari Modul 3 (ringkas)

In [ ]:
kredit = pd.read_csv(DIR / "dataset" / "data_kredit_konsumen.csv")
fitur = ["usia", "penghasilan_juta", "dp_persen",
         "tenor_bulan", "jumlah_telat_sebelumnya"]
X_latih, X_uji, y_latih, y_uji = train_test_split(
    kredit[fitur], kredit["status_pembayaran"],
    test_size=0.2, random_state=42, stratify=kredit["status_pembayaran"])
model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_latih, y_latih)
prediksi = model.predict(X_uji)

## 1. FAKTOR PENDORONG PREDIKSI (FEATURE IMPORTANCE)

In [ ]:
penting = (pd.Series(model.feature_importances_, index=fitur)
           .sort_values(ascending=False))
print()
for nama, nilai in penting.items():
    batang = "#" * int(nilai * 50)
    print(f"  {nama:<26} {nilai:5.1%}  {batang}")

peringkat = list(penting.index)
print(f"""
Cara membacanya untuk bisnis:
  - '{peringkat[0]}' menjadi faktor paling dominan -> indikator ini perlu
    divalidasi sungguh-sungguh saat survei/verifikasi calon konsumen.
  - '{peringkat[1]}' dan '{peringkat[2]}' menyusul -> dapat menjadi dasar
    kebijakan (mis. syarat DP minimum / verifikasi penghasilan).
  - '{peringkat[-1]}' pengaruhnya paling kecil -> jangan jadikan dasar
    keputusan utama.""")

# Grafik feature importance
plt.figure(figsize=(9, 4.5))
penting.sort_values().plot(kind="barh", color="#E60012")
plt.title("Faktor Penentu Risiko Kredit (Feature Importance)")
plt.xlabel("Tingkat kepentingan")
plt.tight_layout()
plt.savefig(DIR / "output" / "modul5_feature_importance.png", dpi=120)
plt.show()

## 2. DARI METRIK TEKNIS KE NILAI BISNIS (RUPIAH)

In [ ]:
cm = confusion_matrix(y_uji, prediksi, labels=["Lancar", "Macet"])
benar_lancar, salah_tolak = cm[0]
lolos_macet, benar_macet = cm[1]

# Asumsi bisnis (ANGKA ILUSTRASI untuk latihan - sesuaikan dengan data riil):
KERUGIAN_PER_MACET = 9_000_000   # estimasi kerugian per kredit macet (Rp)
MARGIN_PER_KONSUMEN = 1_500_000  # margin per konsumen lancar (Rp)

rugi_dicegah = benar_macet * KERUGIAN_PER_MACET
peluang_hilang = salah_tolak * MARGIN_PER_KONSUMEN
rugi_lolos = lolos_macet * KERUGIAN_PER_MACET

print(f"""
Dari {len(y_uji)} pengajuan pada data uji, model:
  - Mendeteksi {benar_macet} calon berisiko macet
      -> potensi kerugian DICEGAH : Rp{rugi_dicegah:,.0f}
  - Salah menolak {salah_tolak} calon yang sebenarnya lancar
      -> peluang margin hilang    : Rp{peluang_hilang:,.0f}
  - Meloloskan {lolos_macet} calon yang ternyata macet
      -> kerugian yang masih lolos: Rp{rugi_lolos:,.0f}

  Estimasi manfaat bersih model  : Rp{rugi_dicegah - peluang_hilang:,.0f}
  (vs tanpa model: semua pengajuan diproses dengan risiko penuh)

Insight: karena kerugian per kredit macet jauh lebih besar daripada margin
per konsumen, model boleh sedikit "lebih galak" menandai risiko -> ambang
keputusan (threshold) bisa disetel sesuai selera risiko manajemen.""")

## 3. RINGKASAN EKSEKUTIF OTOMATIS (SIAP DIKIRIM KE MANAJEMEN)

In [ ]:
akurasi = (prediksi == y_uji).mean()
f1, f2, f3 = [p.replace("_", " ") for p in peringkat[:3]]
f_akhir = peringkat[-1].replace("_", " ")

ringkasan = f"""
RINGKASAN EKSEKUTIF - MODEL PREDIKSI RISIKO KREDIT
---------------------------------------------------
1. Model memprediksi status pembayaran dengan akurasi {akurasi:.0%}
   pada {len(y_uji)} data uji yang belum pernah dilihat model.
2. Faktor risiko terkuat: {f1}, disusul {f2} dan {f3}.
   Faktor {f_akhir} praktis tidak berpengaruh.
3. Estimasi dampak: potensi kerugian Rp{rugi_dicegah / 1e6:,.0f} juta dapat
   dicegah, dengan trade-off peluang hilang Rp{peluang_hilang / 1e6:,.0f} juta.
4. Rekomendasi:
   a. Uji coba model sebagai ALAT BANTU skrining awal (bukan pengganti
      analis) selama 3 bulan.
   b. Tinjau kebijakan DP minimum untuk segmen berisiko.
   c. Evaluasi ulang model tiap kuartal dengan data terbaru.

BATASAN (wajib dicantumkan):
   - Model belajar dari pola masa lalu; perubahan kondisi ekonomi dapat
     menurunkan akurasinya (perlu monitoring berkala).
   - Korelasi bukan sebab-akibat: model menunjukkan ASOSIASI, bukan
     menjelaskan MENGAPA konsumen macet.
   - Keputusan akhir tetap pada manusia; model adalah alat bantu.
"""
print(ringkasan)

file_ringkasan = DIR / "output" / "ringkasan_eksekutif.txt"
file_ringkasan.write_text(ringkasan, encoding="utf-8")
print(f"Ringkasan disimpan ke: {file_ringkasan.name}")
print("Grafik disimpan ke   : modul5_feature_importance.png")

print("""
--------------------------------------------------------------------------------
KESIMPULAN MODUL 5
--------------------------------------------------------------------------------
1. Jangan berhenti di "akurasi 86%" - terjemahkan ke rupiah, risiko, dan
   rekomendasi tindakan. Itulah bahasa pengambil keputusan.
2. Feature importance mengubah model dari "kotak hitam" menjadi dasar
   kebijakan yang bisa dipertanggungjawabkan.
3. Selalu sampaikan BATASAN model. Kejujuran tentang batasan justru
   meningkatkan kepercayaan manajemen terhadap analisis data.
--------------------------------------------------------------------------------
""")

---
### 💡 Latihan mandiri
1. Ubah asumsi bisnis (kerugian per kredit macet / margin per konsumen) sesuai perkiraan di area kerja Anda, lalu lihat perubahan manfaat bersihnya.
2. Tulis ulang ringkasan eksekutif dengan kata-kata Anda sendiri dalam 5 kalimat — latihan inti modul ini adalah *komunikasi*.